In [118]:
import torch

print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print("Using:", device)

PyTorch: 2.14.1
MPS available: True
Using: mps


In [119]:
import numpy as np


def initial_state():
    board = np.zeros((3, 3), dtype=int)  # 提示：np.zeros((行数, 列数), dtype=int)
    current_player = 1
    return board, current_player


# board, player = initial_state()

# board[1, 1] = 1   # 把当前玩家的棋子放到中央
# player = -1            # 切换到另一位玩家

# for row, col in [(1, 1), (0, 0)]:
#     board[row, col] = player   # 哪个变量存着这一手的棋子值？
#     player = -player            # 怎样把当前值的正负号翻转？

# for row, col in [(1, 1), (1, 1)]:
#     if board[row, col] == 0:
#         board[row, col] = player
#         player = -player

In [120]:
def legal_actions(board):
    moves = list()
    for row in range(3):
        for col in range(3):
            if board[row, col] == 0:
                moves.append((row, col))
    return moves

In [121]:
def has_row_win(board, player):
    for row in range(3):
        if np.all(board[row, :] == player):
            return True
    return False

In [122]:
def has_col_win(board, player):
    for col in range(3):
        if np.all(board[:, col] == player):
            return True
    return False

In [123]:
# def has_diag_win(board, player):
#     for i in np.diag(board):
#         if i != player:
#             for i in np.diag(np.fliplr(board)):
#                 if i != player:
#                     return False
#     return True
def has_diag_win(board, player):
    main_win = np.all(np.diag(board) == player)
    anti_win = np.all(np.diag(np.fliplr(board)) == player)
    return main_win or anti_win

In [124]:
def has_won(board, player):
    row = has_row_win(board, player)
    col = has_col_win(board, player)
    diag = has_diag_win(board, player)
    return row or col or diag

In [125]:
def game_result(board, player):
    if has_won(board, -player) == True:
        return -player
    elif len(legal_actions(board)) == 0:
        return 0
    return None

In [126]:
def step(board, player, row, col):
    result = game_result(board, player)
    if result != None:
        return board, player
    if 0 <= row < 3 and 0 <= col < 3 and board[row, col] == 0:
        board[row, col] = player
        player = -player
    return board, player

In [127]:
import random


def random_action(board):
    moves = random.choice(legal_actions(board))
    return moves

In [128]:
def play_random_game():
    board, player = initial_state()

    while game_result(board, player) is None:
        row, col = random_action(board)
        board, player = step(board, player, row, col)

    print(board)
    return game_result(board, player)

In [129]:
# 终局评分
def terminal_value(result, player):
    if result == 0:
        return 0
    elif result == player:
        return 1
    else:
        return -1

In [130]:
# 从当前局面出发，假设双方都走得最好，轮到行动的玩家最终能拿几分？当棋盘上有好几个空格时，这个函数就更有用了。它会试走每个合法位置，再考虑对手会怎样应对。每条走法一旦走到终局，你写的 terminal_value() 就能给它打分。
def minimax_value(board, player):
    result = game_result(board, player)
    if result is not None:
        return terminal_value(result, player)
    scores = []
    for row, col in legal_actions(board):  # 是否有问题，遍历怎么能称为“走得最好”
        child_board, next_player = step(board.copy(), player, row, col)
        child_score = minimax_value(child_board, next_player)
        scores.append(-child_score)
    return max(scores)

In [131]:
def minimax_action(board, player):
    moves = legal_actions(board)
    scores = []
    for row, col in moves:
        child_board, next_player = step(board.copy(), player, row, col)
        child_score = minimax_value(child_board, next_player)
        scores.append(-child_score)

    best_index = scores.index(max(scores))
    return moves[best_index]

In [132]:
def play_minimax_vs_random():
    board, player = initial_state()

    while game_result(board, player) is None:
        if player == -1:
            row, col = minimax_action(board, player)  # O 用 Minimax
        else:
            row, col = random_action(board)  # X 随机选
        board, player = step(board, player, row, col)

    print(board)
    return game_result(board, player)

In [133]:
def play_minimax_vs_minimax():
    board, player = initial_state()

    while game_result(board, player) is None:
        row, col = minimax_action(board, player)
        board, player = step(board, player, row, col)

    print(board)
    return game_result(board, player)

MTCS

In [134]:
def rollout(board, player):
    board = board.copy()
    start_player = player

    while game_result(board, player) is None:
        row, col = random_action(board)
        board, player = step(board, player, row, col)

    result = game_result(board, player)

    return terminal_value(result, start_player)

In [157]:
board99 = np.array([
    [ 1, 0,  1],  # X . X
    [ 1, 1, -1],  # X X O
    [-1, -1, 0],  # O O .
])
N =0
W =0
for i in range(20):
    N=N+1
    W=W+rollout(board99,-1)
Q=W/N
print(N, W, Q)

20 6 0.3
